# Synthetic Microbiome Diversity Analysis After Antibiotic Exposure

As the bioinformatics scientist for project `SYN_MICROBIOME_ABX_001`, I analyze a compact longitudinal gut microbiome cohort exposed to a short antibiotic course. The synthetic design contains 12 subjects sampled at `baseline`, `post_antibiotic`, and `followup`, yielding 36 total stool samples.

Subject IDs use the `SYN-ABX-###` pattern; key taxa include `Bacteroides`, `Faecalibacterium`, `Enterococcus`, and `Akkermansia`. Summary metrics include `post_antibiotic_delta` and `recovered_subjects`.

In [2]:
import random

SEED = 271828
rng = random.Random(SEED)

PROJECT_ID = "SYN_MICROBIOME_ABX_001"
subject_ids = [f"SYN-ABX-{i:03d}" for i in range(1, 13)]
timepoints = ["baseline", "post_antibiotic", "followup"]

sample_metadata = []
for subject_id in subject_ids:
    for timepoint in timepoints:
        sample_metadata.append(
            {
                "project_id": PROJECT_ID,
                "subject_id": subject_id,
                "timepoint": timepoint,
                "sample_id": f"{subject_id}_{timepoint}",
            }
        )

print(f"project_id {PROJECT_ID}")
print(f"random_seed {SEED}")
print(f"subjects {len(subject_ids)}")
print(f"samples {len(sample_metadata)}")
print(f"timepoints {', '.join(timepoints)}")
print("\nFirst six sample records:")
for row in sample_metadata[:6]:
    print(f"{row['sample_id']:<28} {row['subject_id']:<12} {row['timepoint']}")

project_id SYN_MICROBIOME_ABX_001
random_seed 271828
subjects 12
samples 36
timepoints baseline, post_antibiotic, followup

First six sample records:
SYN-ABX-001_baseline         SYN-ABX-001  baseline
SYN-ABX-001_post_antibiotic  SYN-ABX-001  post_antibiotic
SYN-ABX-001_followup         SYN-ABX-001  followup
SYN-ABX-002_baseline         SYN-ABX-002  baseline
SYN-ABX-002_post_antibiotic  SYN-ABX-002  post_antibiotic
SYN-ABX-002_followup         SYN-ABX-002  followup


## Synthetic Cohort And Marker Panel

The taxon panel is deliberately small for inspectability. I modeled a typical antibiotic response: obligate anaerobes such as `Faecalibacterium` and `Roseburia` contract immediately after exposure, `Enterococcus` blooms, and a subset of subjects partially reassemble a baseline-like community by follow-up.

Eight subjects are assigned a recovered follow-up profile by the fixed random seed; the other four retain an elevated `Enterococcus` signal.

In [3]:
taxa = [
    "marker_ASV_Bacteroides_uniformis",
    "marker_ASV_Faecalibacterium_prausnitzii",
    "marker_ASV_Enterococcus_faecalis",
    "marker_ASV_Akkermansia_muciniphila",
    "marker_ASV_Blautia_wexlerae",
    "marker_ASV_Roseburia_intestinalis",
    "marker_ASV_Escherichia_coli",
    "marker_ASV_Ruminococcus_bromii",
]

profiles = {
    "baseline": [2300, 2200, 700, 1200, 1100, 1000, 800, 700],
    "post_antibiotic": [1010, 404, 6970, 202, 505, 202, 505, 202],
    "followup_recovered": [2100, 2000, 900, 1300, 1100, 1000, 900, 700],
    "followup_not_recovered": [1400, 700, 5000, 300, 800, 400, 900, 500],
}

recovered_followup_subject_ids = set(rng.sample(subject_ids, 8))
abundance_table = {}
for row in sample_metadata:
    profile_name = row["timepoint"]
    if row["timepoint"] == "followup":
        profile_name = "followup_recovered" if row["subject_id"] in recovered_followup_subject_ids else "followup_not_recovered"
    row["profile_name"] = profile_name
    abundance_table[row["sample_id"]] = dict(zip(taxa, profiles[profile_name]))

print(f"taxa_markers {len(taxa)}")
print(f"abundance_table_shape {len(abundance_table)} samples x {len(taxa)} taxa")
print("recovered_followup_subject_ids " + ", ".join(sorted(recovered_followup_subject_ids)))
print("\nSelected marker counts for SYN-ABX-001:")
print("sample_id                     Bacteroides  Faecalibacterium  Enterococcus  Akkermansia")
for sample_id in ["SYN-ABX-001_baseline", "SYN-ABX-001_post_antibiotic", "SYN-ABX-001_followup"]:
    counts = abundance_table[sample_id]
    print(
        f"{sample_id:<31}"
        f"{counts['marker_ASV_Bacteroides_uniformis']:>8}"
        f"{counts['marker_ASV_Faecalibacterium_prausnitzii']:>18}"
        f"{counts['marker_ASV_Enterococcus_faecalis']:>14}"
        f"{counts['marker_ASV_Akkermansia_muciniphila']:>13}"
    )

taxa_markers 8
abundance_table_shape 36 samples x 8 taxa
recovered_followup_subject_ids SYN-ABX-001, SYN-ABX-002, SYN-ABX-003, SYN-ABX-006, SYN-ABX-007, SYN-ABX-009, SYN-ABX-010, SYN-ABX-011

Selected marker counts for SYN-ABX-001:
sample_id                     Bacteroides  Faecalibacterium  Enterococcus  Akkermansia
SYN-ABX-001_baseline               2300              2200           700         1200
SYN-ABX-001_post_antibiotic        1010               404          6970          202
SYN-ABX-001_followup               2100              2000           900         1300


## Diversity And Distance Functions

I use Shannon diversity as the alpha-diversity readout and Bray-Curtis distance as a simple abundance-sensitive beta-diversity measure. Shannon diversity is computed with `calculate_shannon_index`.

In [4]:
import math


def calculate_shannon_index(counts):
    """Return Shannon diversity using natural-log proportions."""
    total = sum(counts)
    proportions = [count / total for count in counts if count > 0]
    return -sum(p * math.log(p) for p in proportions)


def bray_curtis_distance(left_counts, right_counts):
    numerator = sum(abs(a - b) for a, b in zip(left_counts, right_counts))
    denominator = sum(a + b for a, b in zip(left_counts, right_counts))
    return numerator / denominator if denominator else 0.0


for row in sample_metadata:
    row_counts = [abundance_table[row["sample_id"]][taxon] for taxon in taxa]
    row["shannon"] = calculate_shannon_index(row_counts)

summary_rows = []
for timepoint in timepoints:
    values = [row["shannon"] for row in sample_metadata if row["timepoint"] == timepoint]
    summary_rows.append((timepoint, len(values), sum(values) / len(values), min(values), max(values)))

print("Shannon diversity summary by timepoint")
print("timepoint          n  mean_shannon  min_shannon  max_shannon")
for timepoint, n_values, mean_value, min_value, max_value in summary_rows:
    print(f"{timepoint:<16}{n_values:>3}{mean_value:>14.3f}{min_value:>13.3f}{max_value:>13.3f}")

Shannon diversity summary by timepoint
timepoint          n  mean_shannon  min_shannon  max_shannon
baseline         12         1.973        1.973        1.973
post_antibiotic  12         1.151        1.151        1.151
followup         12         1.875        1.610        2.007


## Longitudinal Diversity Comparison

For the primary comparison, I calculate subject-level baseline, post-antibiotic, and follow-up Shannon values. A subject is counted as recovered if follow-up Shannon diversity is within 0.20 Shannon units of that subject's baseline value.

In [5]:
shannon_by_subject = {
    row["subject_id"]: {} for row in sample_metadata
}
for row in sample_metadata:
    shannon_by_subject[row["subject_id"]][row["timepoint"]] = row["shannon"]

baseline_mean = sum(values["baseline"] for values in shannon_by_subject.values()) / len(shannon_by_subject)
post_mean = sum(values["post_antibiotic"] for values in shannon_by_subject.values()) / len(shannon_by_subject)
followup_mean = sum(values["followup"] for values in shannon_by_subject.values()) / len(shannon_by_subject)

post_antibiotic_delta = round(post_mean - baseline_mean, 2)
followup_delta_from_baseline = round(followup_mean - baseline_mean, 2)
recovered_subjects = sum(
    1
    for values in shannon_by_subject.values()
    if values["followup"] >= values["baseline"] - 0.20
)

print(f"baseline_mean_shannon {baseline_mean:.2f}")
print(f"post_antibiotic_mean_shannon {post_mean:.2f}")
print(f"followup_mean_shannon {followup_mean:.2f}")
print(f"post_antibiotic_delta {post_antibiotic_delta:.2f}")
print(f"followup_delta_from_baseline {followup_delta_from_baseline:.2f}")
print(f"recovered_subjects {recovered_subjects}")
print("\nInterpretation: the antibiotic exposure creates a strong alpha-diversity collapse, while follow-up shows partial recovery rather than complete return to baseline.")

baseline_mean_shannon 1.97
post_antibiotic_mean_shannon 1.15
followup_mean_shannon 1.88
post_antibiotic_delta -0.82
followup_delta_from_baseline -0.10
recovered_subjects 8

Interpretation: the antibiotic exposure creates a strong alpha-diversity collapse, while follow-up shows partial recovery rather than complete return to baseline.


In [6]:
sample_ids = [row["sample_id"] for row in sample_metadata]
count_vectors = [
    [abundance_table[sample_id][taxon] for taxon in taxa]
    for sample_id in sample_ids
]
bray_curtis_matrix = [
    [bray_curtis_distance(left, right) for right in count_vectors]
    for left in count_vectors
]

print(f"Bray-Curtis distance matrix size {len(bray_curtis_matrix)} x {len(bray_curtis_matrix[0])}")
print("Bray-Curtis preview for first five samples")
preview_ids = sample_ids[:5]
print("sample_id".ljust(33) + "".join(sample_id.rjust(30) for sample_id in preview_ids))
for i, sample_id in enumerate(preview_ids):
    values = "".join(f"{bray_curtis_matrix[i][j]:>30.3f}" for j in range(5))
    print(f"{sample_id:<33}{values}")

ordination_coordinates = []
for row in sample_metadata:
    counts = abundance_table[row["sample_id"]]
    total = sum(counts.values())
    ordination_coordinates.append(
        {
            "sample_id": row["sample_id"],
            "timepoint": row["timepoint"],
            "PCo1_proxy": counts["marker_ASV_Enterococcus_faecalis"] / total,
            "PCo2_proxy": row["shannon"],
        }
    )

print("\nordination_plot_placeholder synthetic_pcoa_enterococcus_axis")
print("PCo1 proxy: Enterococcus relative abundance; PCo2 proxy: Shannon diversity")
print("baseline cluster: low PCo1 / high PCo2")
print("post_antibiotic cluster: high PCo1 / low PCo2")
print("followup cluster: intermediate PCo1 / partially restored PCo2")
print("\nOrdination coordinate preview")
print("sample_id".ljust(32) + "timepoint".ljust(20) + "PCo1_proxy  PCo2_proxy")
for row in ordination_coordinates[:6]:
    print(f"{row['sample_id']:<32}{row['timepoint']:<20}{row['PCo1_proxy']:>6.3f}{row['PCo2_proxy']:>12.3f}")

Bray-Curtis distance matrix size 36 x 36
Bray-Curtis preview for first five samples
sample_id                                  SYN-ABX-001_baseline   SYN-ABX-001_post_antibiotic          SYN-ABX-001_followup          SYN-ABX-002_baseline   SYN-ABX-002_post_antibiotic
SYN-ABX-001_baseline                                      0.000                         0.627                         0.040                         0.000                         0.627
SYN-ABX-001_post_antibiotic                               0.627                         0.000                         0.607                         0.627                         0.000
SYN-ABX-001_followup                                      0.040                         0.607                         0.000                         0.040                         0.607
SYN-ABX-002_baseline                                      0.000                         0.627                         0.040                         0.000                         0.

## Conclusion

The synthetic longitudinal gut cohort shows the expected antibiotic perturbation: Shannon diversity drops sharply at `post_antibiotic`, driven by an `Enterococcus` bloom and depletion of anaerobic markers such as `Faecalibacterium`. By `followup`, 8 of 12 subjects meet the recovery threshold, but the cohort mean remains slightly below baseline.

My conclusion is that the community demonstrates partial recovery at follow-up, not full restoration. Antibiotic-associated disruption shows alpha-diversity loss, Bray-Curtis separation from baseline, and incomplete recovery by follow-up for `SYN_MICROBIOME_ABX_001`.